## Thorlabs Spectrometer. Data Processing Part 1

**Dataset:** `\2026_09_11_spec\p4p5Pa_fast_scan` 

**Steps:** Background subtraction $\rightarrow$ Exposure time normalization $\rightarrow$ Export


#### Output Data Structures
1. **pixel intensity** *as function of power or index of mesurements* 
- spectr_A: spectrometer near RF antenna
- spectr_B: backside spectrometer

2. **spectrometer wavelength** *vs pixel number*
- wavelength map 

In [ ]:
from pathlib import Path
import sys
import pandas as pd
##### import project related moduls ####
current_file = Path.cwd() # cwd = path/*.ipynb - does not work in .py files.
print(f"current_file = {current_file}")
project_root = current_file.parent.parent
print(f"project_root = {project_root}")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import utils.file_utils as fu   

In [ ]:
folder_path=Path(r"C:\Andrei\DATA\VINETA_75\2026_09_11_spec\p4p5Pa_fast_scan\txt")
files = fu.list_files_in_folder(folder_path=folder_path)
list_name_parts = []
for file in files:
    # print(file.name)
    name_parts = file.name.split("_")
    print(name_parts)
    list_name_parts.append(name_parts)
    # name_dict = {
    #     "index": name_parts[0],
    #     "power"
    # }

In [ ]:
## load background file with 2 seconds integration time for spectrometer B
print(files[-2].name)
df_bg_2s = pd.read_csv(files[-2], sep="\t", header=1, 
                       names=["wavelength1", "intensity1",
                            "wavelength2", "intensity2"],
                        dtype={"wavelength1": float, "intensity1": float,
                               "wavelength2": float, "intensity2": float},
                            )
print(df_bg_2s.head())
print(df_bg_2s.info())


In [ ]:
print(f"{folder_path=}")
output_folder = folder_path.parent / "bg_corr"
print(f"{output_folder=}")
output_folder.mkdir(exist_ok=True)

In [ ]:
## substract backgrond from first 9 files with txB=2s and normalize on tx. 
## save results in folder bg_corr 
i = 0
for file in files[:9]:
    print(file.name)
    df = pd.read_csv(file, sep="\t", header=1, 
                     names=["wavelength1", "intensity1", 
                             "wavelength2", "intensity2"],
                    dtype={"wavelength1": float, "intensity1": float, 
                            "wavelength2": float, "intensity2": float})
    
    # print(df.info())

    df_substracted = df.copy()
    df_substracted["intensity1"] = (df["intensity1"] - df_bg_2s["intensity1"]) / 0.2
    df_substracted["intensity2"] = (df["intensity2"] - df_bg_2s["intensity2"]) / 2
    # print(df_substracted.info())
    # print(df_substracted.head())
    name_parts = list_name_parts[i]
    i += 1
    print(name_parts)

    new_file_name = name_parts[0] + "_" + name_parts[1] + ".txt"
    new_file_path = output_folder / new_file_name
    print(f"{new_file_path=}")
    df_substracted.to_csv(new_file_path, sep="\t", index=False)


In [ ]:
## load background file with 1 second integration time for spectrometer B
print(files[-1].name)
df_bg_1s = pd.read_csv(files[-2], sep="\t", header=1, 
                       names=["wavelength1", "intensity1",
                            "wavelength2", "intensity2"],
                        dtype={"wavelength1": float, "intensity1": float,
                               "wavelength2": float, "intensity2": float},
                            )
print(df_bg_1s.head())
print(df_bg_1s.info())

In [ ]:
print(f"{len(files)=}")
print(f"{files[26:28]=}")
print(f"{files[9]=}")
print(f"{files[25]=}")

In [ ]:
## substract backgrond from first 9 files with txB=1s and normalize on tx. 
## save results in folder bg_corr 
i = 9
for file in files[9:26]:
    print(file.name)
    df = pd.read_csv(file, sep="\t", header=1, 
                     names=["wavelength1", "intensity1", 
                             "wavelength2", "intensity2"],
                    dtype={"wavelength1": float, "intensity1": float, 
                            "wavelength2": float, "intensity2": float})
    
    # print(df.info())

    df_substracted = df.copy()
    df_substracted["intensity1"] = (df["intensity1"] - df_bg_1s["intensity1"]) / 0.2
    df_substracted["intensity2"] = (df["intensity2"] - df_bg_1s["intensity2"] )/ 1
    # print(df_substracted.info())
    # print(df_substracted.head())
    name_parts = list_name_parts[i]
    i += 1
    print(name_parts)

    new_file_name = name_parts[0] + "_" + name_parts[1] + ".txt"
    new_file_path = output_folder / new_file_name
    print(f"{new_file_path=}")
    df_substracted.to_csv(new_file_path, sep="\t", index=False)

In [ ]:

folder_path = output_folder
print(folder_path)
files = fu.list_files_in_folder(folder_path=folder_path)
list_name_parts = []
data_A = {}
data_B = {}
wavelengths_map = None
for file in files:
    # print(file.name)
    ## infer power values from flie name
    name_parts = file.name.split("_")
    power = name_parts[-1].replace("p", ".").replace(".txt", "").replace("P", "")
    power = float(power)
    index = int(name_parts[0])
    print(name_parts)
    print(index, power)
    list_name_parts.append(name_parts)

    df = pd.read_csv(file, sep="\t", header=0)
    ## set wavlength map for the first file only
    if wavelengths_map is None:
        wavelengths_map = pd.DataFrame({
            "pixel": df.index,
            "wavelength1": df["wavelength1"], 
            "wavelength2": df["wavelength2"]
            })
    row_A = df["intensity1"].copy()
    row_A["power"] = power
    data_A[index] = row_A

    row_B = df["intensity2"].copy()
    row_B["power"] = power
    data_B[index] = row_B


In [ ]:
spectr_A = pd.DataFrame.from_dict(data_A, orient="index").sort_index()
spectr_A.insert(0, "power", spectr_A.pop("power"))
# print(spectr_A.head(2))

spectr_B = pd.DataFrame.from_dict(data_B, orient="index").sort_index()
spectr_B.insert(0, "power", spectr_B.pop("power"))
print(spectr_B.head(3))

print(wavelengths_map.head(3))

In [ ]:
print(folder_path)
output_folder = folder_path.parent / "tables"
print(output_folder)
output_folder.mkdir(exist_ok=True)


In [ ]:
table_A_path = output_folder / "spectr_A.csv"
table_B_path = output_folder / "spectr_B.csv"
table_map_path = output_folder / "wavelengths_map.csv"

spectr_A.to_csv(table_A_path, sep="\t", index=True)
spectr_B.to_csv(table_B_path, sep="\t", index=True)
wavelengths_map.to_csv(table_map_path, sep="\t", index=True)